In [2]:
from src.dataset_manager import DatasetManager
import pandas as pd

In [5]:
def detectar_outliers_robust(lista_ids, factor_iqr=2.0):
    reporte = []
    
    for idx in lista_ids:
        df_motor = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
        columnas_datos = [c for c in df_motor.columns if c not in ['unit_number', 'time_in_cycles', 'RUL', 'evento']]
        
        for col in columnas_datos:
            serie = df_motor[col]
            q1 = serie.quantile(0.25)
            q3 = serie.quantile(0.75)
            iqr = q3 - q1
            mediana = serie.median()
            
            if iqr > 0:
                # Escalado robusto manual
                serie_robust = (serie - mediana) / iqr
                
                indices = serie_robust[serie_robust.abs() > factor_iqr].index
                
                for i in indices:
                    reporte.append({
                        'unit_number': idx,
                        'ciclo': df_motor.loc[i, 'time_in_cycles'],
                        'columna': col,
                        'valor_robust': serie_robust[i]
                    })
                    
    return pd.DataFrame(reporte)


In [8]:
metadata = pd.read_csv('data/metadata.csv')
m_train, m_test = DatasetManager.split_dataset(metadata)

Motores para entrenamiento: 140
Motores para prueba: 60


In [6]:
df_outliers_robust = detectar_outliers_robust(m_train, factor_iqr=3.0)
display(df_outliers_robust)

,unit_number,ciclo,columna,valor_robust
0,98,148,NRc,-3.050232
1,98,155,NRc,-3.413255
2,98,156,NRc,-4.179823
3,13,159,T30,3.280985
4,13,159,Nc,3.365118
...,...,...,...,...
756,15,203,htBleed,4.000000
757,15,204,htBleed,4.000000
758,15,205,htBleed,5.000000
759,15,206,htBleed,4.000000


In [7]:
def resumir_outliers_por_motor(df_outliers, m_train_ids):
    # 1. Agrupar por motor y columna para contar cuántos ciclos atípicos hay
    resumen = df_outliers.groupby(['unit_number', 'columna']).agg(
        num_outliers=('ciclo', 'count'),
        ciclo_inicio=('ciclo', 'min'),
        ciclo_fin=('ciclo', 'max'),
        max_desviacion=('valor_robust', lambda x: x.abs().max())
    ).reset_index()
    
    # 2. Calcular la proporción respecto al total de vida de cada motor
    # Necesitamos saber cuántas filas totales tiene cada motor original
    total_filas = {}
    for idx in m_train_ids:
        # Cargamos solo una vez para eficiencia
        df_temp = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
        total_filas[idx] = len(df_temp)
    
    resumen['total_ciclos_motor'] = resumen['unit_number'].map(total_filas)
    resumen['proporcion_atipica'] = (resumen['num_outliers'] / resumen['total_ciclos_motor']) * 100
    
    return resumen.sort_values(by='proporcion_atipica', ascending=False)

# Ejecutar el resumen
df_resumen_atipicos = resumir_outliers_por_motor(df_outliers_robust, m_train)

# Ver los 15 casos más críticos
display(df_resumen_atipicos.head(15))

,unit_number,columna,num_outliers,ciclo_inicio,ciclo_fin,max_desviacion,total_ciclos_motor,proporcion_atipica
54,18,htBleed,16,165,195,6.000000,195,8.205128
172,82,Nc,14,201,214,4.556764,214,6.542056
2,3,NRc,11,169,179,3.938573,179,6.145251
101,48,Nc,14,218,231,4.352908,231,6.060606
31,11,NRc,14,227,240,4.001342,240,5.833333
12,4,Nc,11,178,189,4.240192,189,5.820106
44,18,NRc,11,185,195,4.263686,195,5.641026
124,51,htBleed,12,177,213,6.000000,213,5.633803
24,9,Nc,11,190,201,4.385314,201,5.472637
22,9,NRc,11,191,201,4.400919,201,5.472637
